## Anamoly Alphas

In [1]:
import matplotlib.pyplot as plt
from data_loader import load_data, get_close_prices, get_volume, get_open_prices
from alpha_utils import evaluate_alpha
from alpha_utils import ts_rank, ts_min, ts_max, linear_decay
from alpha_utils import rolling_corr
from alpha_utils import SPLIT_DATE 
import numpy as np 
import pandas as pd 
import warnings
from scipy.stats import ConstantInputWarning
warnings.filterwarnings('ignore', category=ConstantInputWarning)

data = load_data()
close = get_close_prices(data)
open_px = get_open_prices(data)
volume = get_volume(data)
n_stocks = close.shape[1]
print(close.shape)

Loading cached data from data/sp500_5y.parquet
(1255, 503)


In [2]:
print(data.columns.get_level_values(1).unique())

Index(['Open', 'High', 'Low', 'Close', 'Volume', 'Adj Close'], dtype='object', name='Price')


## Earnings Surprise Proxy
### Price + Volume Shock. Sharpe : 1.19 (OSS : Train 1.06, Test 1.44)

In [3]:
close = get_close_prices(data)
volume = get_volume(data)

In [4]:
returns = close.pct_change()

/var/folders/76/6k6n689j2g5gzbx7kpxdvjl00000gn/T/ipykernel_4291/3478380900.py:1: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = close.pct_change()


In [5]:
'''returns z score (magnitude of price surprise)'''
n = 10
abs_ret = returns.abs()
ret_mean = abs_ret.rolling(n).mean()
ret_std = abs_ret.rolling(n).std()
ret_z = (abs_ret - ret_mean)/ret_std

In [6]:
'''returns z score (magnitude of volume surprise)'''
n = 10
unitless_vol = volume/ volume.rolling(n).mean()
log_vol = np.log(unitless_vol)
vol_mean = log_vol.rolling(n).mean()
vol_std = log_vol.rolling(n).std()
vol_z = (log_vol - vol_mean)/vol_std

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/internals/blocks.py:393: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)


In [7]:
surprise = np.minimum(ret_z, vol_z)

In [8]:
'''quiet days should contribute zero; no neagtive days'''
surprise_clip = surprise.clip(lower=0)

In [9]:
raw_signal_ESP = -1 * np.sign(returns) * surprise_clip

In [10]:
signal_ESP = linear_decay(raw_signal_ESP, 9)

In [11]:
results_ESP = evaluate_alpha(signal_ESP, close, n_stocks)
print("Sharpe:", results_ESP["sharpe"])
print("IC mean:", results_ESP["ic_mean"])
print("ICIR:", results_ESP["icir"])

Sharpe: 0.5097734134408138
IC mean: 0.01048925784883774
ICIR: 0.07674265625528282


In [12]:
# Test n=200 
avg_dollar_volume = (close * volume).rolling(60).mean().iloc[-1]

small_cap_proxy_200 = avg_dollar_volume.nsmallest(200).index
sig_200 = signal_ESP[small_cap_proxy_200]
cls_200 = close[small_cap_proxy_200]

results_final = evaluate_alpha(sig_200, cls_200, n_stocks = 200, weighting='signal')
print('Final Sharpe:', results_final["sharpe"])
print('Final IC mean:', results_final["ic_mean"])
print('Final ICIR:', results_final["icir"])

Final Sharpe: 1.199705407634608
Final IC mean: 0.014771663612783613
Final ICIR: 0.1066440414314599


## OOS Test (Train) for ESP

In [13]:
sig_200_train = sig_200[sig_200.index < SPLIT_DATE]
cls_200_train = cls_200[cls_200.index < SPLIT_DATE]

results_train = evaluate_alpha(sig_200_train, cls_200_train, n_stocks=200, weighting='signal')
print('train_sharpe:', results_train["sharpe"])
print('train_ic_mean:', results_train["ic_mean"])

train_sharpe: 1.0603290550566136
train_ic_mean: 0.013195828927256426


## OOS Test (Test) for ESP

In [14]:
sig_200_test = sig_200[sig_200.index >= SPLIT_DATE]
cls_200_test = cls_200[cls_200.index >= SPLIT_DATE]

results_test = evaluate_alpha(sig_200_test, cls_200_test, n_stocks=200, weighting='signal')
print('test sharpe:', results_test["sharpe"])
print('test ic:', results_test["ic_mean"])

test sharpe: 1.4439608844423475
test ic: 0.01839187057177711


## Earning Surprise Proxy -- Complete
Signal : price z_score ∩ volume z_score (min), reversal direction : (-1 * sign(returns), decay=9)

Universe : Bottom 200 stocks by 60 avg dollar volume (small-cap proxy within S&P 500)

Weighting : signal strength (not rank bucket)

long_pct/short_pct = 0.10

full sample sharpe = 1.19 | IC mean = 0.014

Train (before 2025-02-06) : Sharpe = 1.06 | IC Mean = 0.013

Test/OOS : Sharpe = 1.44 | IC mean = 0.018
